# 第6周 Day7 · 第六周总复习——Agent 核心机制概念实验

> 本 notebook 是《第6周-Day7-第六周总复习.md》的可执行配套实验。核心公式：**Agent = LLM（大脑）+ Tools（工具）+ Memory（记忆）+ ReAct（循环）**。
>
> 实验仅用 numpy / 标准库 / matplotlib（真实系统中 Thought 由 LLM 生成，这里用规则脚本模拟其决策过程）：
>
> 1. Function Calling：工具定义三要素（Name + Description + Parameters）与**参数校验**——为什么参数错误不重试
> 2. 迷你 ReAct 循环：Thought → Action → Observation 的补货决策全链路
> 3. 记忆分层检索：短期 / 中期 / 长期记忆的命中率与延迟模拟
> 4. 可视化：ReAct 反馈循环 vs 一步直答 + 记忆分层效果

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("实验环境就绪。Agent 三大核心组件（大脑手套记忆法）：")
print("  LLM（大脑）——思考、决策、推理：决定'做什么'")
print("  Tools（手套）——执行操作：'动手做'")
print("  Memory（笔记本）——记录历史：'记住经验'")
print("一句话记忆：大脑指挥、工具干活、记忆帮忙。")

## 实验 1：Function Calling——工具定义与参数校验

工具定义三要素 = **N**ame（动宾结构）+ **D**escription（LLM 选择工具的唯一依据，最重要！）+ **P**arameters（类型 + 必填 + 描述）。

LLM 抽取的参数可能出错，执行前必须校验。关键原则（对应 md 错误处理策略）：
- **参数错误是确定性错误**：同样的输入重试只会得到同样的错误 → 不重试，把错误信息**反馈给 LLM 修正**；
- 网络超时 / 限流等**暂时性错误**才使用指数退避重试。

（教学版手写校验；生产版可用 jsonschema / pydantic，本实验仅标准库。）

In [ ]:
TOOL_SPECS = {
    "query_stock": {
        "description": "查询指定商品当前库存、日均销量与供应商交期",
        "parameters": {
            "product": {"type": str, "required": True, "desc": "商品名称"},
        },
    },
    "calc_restock": {
        "description": "计算补货建议：提前期需求+安全库存 与 当前库存比较",
        "parameters": {
            "stock":       {"type": (int, float), "required": True, "desc": "当前库存"},
            "daily_sales": {"type": (int, float), "required": True, "desc": "日均销量"},
            "lead_days":   {"type": int,          "required": True, "desc": "供应商交期（天）"},
        },
    },
}

def validate_arguments(tool_name, args):
    """执行前的参数校验：未知工具 / 缺必填 / 类型错误 / 语义越界。
    返回错误列表（空列表 = 通过）。"""
    if tool_name not in TOOL_SPECS:
        return [f"未知工具: {tool_name}"]
    errors = []
    spec = TOOL_SPECS[tool_name]["parameters"]
    for pname, pdef in spec.items():                      # 1) 必填检查
        if pdef["required"] and pname not in args:
            errors.append(f"缺少必填参数 '{pname}'")
    for pname, value in args.items():                     # 2) 类型检查
        if pname not in spec:
            errors.append(f"未知参数 '{pname}'")
            continue
        expected = spec[pname]["type"]
        allowed = expected if isinstance(expected, tuple) else (expected,)
        if type(value) not in allowed:                    # 用 type() 避免 bool 冒充 int
            errors.append(f"参数 '{pname}' 类型错误: 期望{allowed}, 实为{type(value).__name__}")
    # 3) 语义检查（类型对了但取值不合理）
    if not errors:
        if tool_name == "query_stock" and not args["product"].strip():
            errors.append("'product' 不能为空字符串")
        if tool_name == "calc_restock":
            if args["lead_days"] < 1:
                errors.append("'lead_days' 需为 ≥1 的整数")
            if args["daily_sales"] < 0 or args["stock"] < 0:
                errors.append("'daily_sales'/'stock' 不能为负数")
    return errors

tests = [
    ("query_stock",  {"product": "柠檬茶"}, True),
    ("query_stock",  {}, False),                                            # 缺必填
    ("query_stock",  {"product": 123}, False),                              # 类型错
    ("query_stock",  {"product": "  "}, False),                             # 语义错
    ("calc_restock", {"stock": 95, "daily_sales": 18, "lead_days": 4}, True),
    ("calc_restock", {"stock": "95", "daily_sales": 18, "lead_days": 4}, False),  # 类型错
    ("calc_restock", {"stock": 95, "daily_sales": -1, "lead_days": 0}, False),    # 语义错
    ("get_weather",  {"city": "广州"}, False),                              # 未知工具
]

n_pass = 0
for tool, args, expect_ok in tests:
    errs = validate_arguments(tool, args)
    passed = not errs
    n_pass += (passed == expect_ok)
    verdict = "通过" if passed else "拒绝→反馈给LLM修正(不重试): " + "; ".join(errs)
    print(f"[{'通过' if passed else '拒绝'}] {tool}({args})\n        → {verdict}")
print(f"\n校验器行为符合预期: {n_pass}/{len(tests)}")

## 实验 2：迷你 ReAct Agent——Thought → Action → Observation

任务：*"柠檬茶要不要补货？"*（对应 md 业务案例）

教学版要点：
- **Thought** 由规则脚本模拟（生产版由 LLM 生成）：先查事实，再做透明计算，证据齐全才下结论；
- **Action** 执行前经过实验 1 的参数校验——校验失败即终止并反馈，而不是带着脏参数行动；
- **Observation** 写回上下文，驱动下一步决策（不是一条路走到黑）；
- 轨迹存入 `history`（Memory 的最简形态），全程可审计。

In [ ]:
SHOP = {
    "柠檬茶":   {"stock": 95, "daily_sales": 18, "lead_days": 4},
    "杨枝甘露": {"stock": 30, "daily_sales": 25, "lead_days": 2},
}

def query_stock(product):
    info = SHOP[product]
    return {"product": product, **info}

def calc_restock(stock, daily_sales, lead_days):
    lead_demand = daily_sales * lead_days      # 到货前预计消耗
    safety_stock = daily_sales * 2             # 安全库存（简化：2天销量）
    return {
        "lead_demand": lead_demand, "safety_stock": safety_stock,
        "threshold": lead_demand + safety_stock,
        "reorder": stock < lead_demand + safety_stock,
        "days_cover": round(stock / daily_sales, 1),
    }

TOOLS = {"query_stock": query_stock, "calc_restock": calc_restock}

class MiniReActAgent:
    """最小可运行 ReAct：T(hought)-A(ction)-O(observation) 循环 + 短期记忆。"""

    def __init__(self, tools, max_steps=5):
        self.tools = tools
        self.max_steps = max_steps
        self.history = []                      # Memory：轨迹记忆

    def _think(self, goal, obs):
        """模拟 LLM 的决策：根据已有观察决定下一步（教学版=规则）。"""
        if "事实" not in obs:                  # 第一层：先取证，不凭记忆回答
            return "用户问补货，先查库存/销量/交期事实", \
                   "query_stock", {"product": "柠檬茶"}
        if "计算" not in obs:                  # 第二层：用可解释公式计算
            f = obs["事实"]
            return "用 提前期需求+安全库存 与当前库存比较", \
                   "calc_restock", {"stock": f["stock"], "daily_sales": f["daily_sales"],
                                    "lead_days": f["lead_days"]}
        return None                             # 证据齐全 → 输出结论

    def run(self, goal):
        print(f"目标: {goal}\n" + "-" * 60)
        obs, trace = {}, []
        for step in range(1, self.max_steps + 1):
            decision = self._think(goal, obs)
            if decision is None:
                calc = obs["计算"]
                print(f"[Thought {step}]   证据齐全，输出结论并说明原因")
                print(f"[Final]   {'建议补货' if calc['reorder'] else '暂不补货'}："
                      f"库存95 < 提前期需求{calc['lead_demand']} + 安全库存{calc['safety_stock']}"
                      f"（可覆盖 {calc['days_cover']} 天）")
                trace.append(("final", calc))
                break
            thought, action, args = decision
            errs = validate_arguments(action, args)   # Action 前强制校验
            if errs:
                print(f"[Thought {step}]   参数校验失败，终止并反馈: {errs}")
                break
            print(f"[Thought {step}]   {thought}")
            print(f"[Action {step}]    {action}({args})")
            result = self.tools[action](**args)
            obs["事实" if action == "query_stock" else "计算"] = result
            trace.append((action, result))
            print(f"[Observation]     {result}\n")
        self.history.append({"goal": goal, "trace": trace})
        return trace

agent = MiniReActAgent(TOOLS)
trace = agent.run("柠檬茶要不要补货？")
print(f"\n本轨迹共 {len(trace)} 步（含取事实、计算、结论），已写入 agent.history 供审计。")

## 实验 3：记忆分层——短期 / 中期 / 长期的检索模拟

对应 md"短中长法则"：**对话用内存、用户用缓存、知识用数据库**。

- **短期**（内存，最近 8 轮）：直接命中，延迟 ~0.1ms；
- **中期**（Redis，TTL 内）：命中，~1.2ms；
- **长期**（向量库）：用余弦相似度检索 top-k，~9ms，且**可能检索失败**（相似条目干扰）→ 未命中需人工兜底（~50ms）。

模拟 400 次查询，目标记忆距"上次使用"的轮龄均匀分布于 1~89 轮。

In [ ]:
rng = np.random.default_rng(7)
DIM = 24
THEMES = ["口味偏好", "历史订单", "投诉记录", "会员等级", "营业时段", "过敏信息"]

def normalize(x):
    x = np.asarray(x, dtype=float)
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

# 长期记忆条目：主题向量 + 噪声（模拟 embedding：同主题相近、不同主题远离）
theme_vec = normalize(rng.normal(size=(len(THEMES), DIM)))
long_items = []
for i in range(60):
    t = i % len(THEMES)
    vec = normalize(theme_vec[t] + 0.45 * rng.normal(size=DIM))
    long_items.append({"id": i, "theme": THEMES[t], "vec": vec})

SHORT_WINDOW, MID_WINDOW = 8, 24          # 短期=最近8轮；中期=TTL=24轮
TIER_COST = {"短期": 0.1, "中期": 1.2, "长期": 9.0, "未命中": 50.0}

def layered_retrieve(age_turns, target_item, top_k=3):
    """按 短期→中期→长期 逐层检索，返回 (结果层, 延迟ms)。"""
    if age_turns <= SHORT_WINDOW:
        return "短期", TIER_COST["短期"]
    if age_turns <= MID_WINDOW:
        return "中期", TIER_COST["中期"]
    query = normalize(target_item["vec"] + 0.6 * rng.normal(size=DIM))  # 带噪查询
    sims = np.array([np.dot(query, it["vec"]) for it in long_items])
    top = np.argsort(-sims)[:top_k]
    return (("长期", TIER_COST["长期"]) if target_item["id"] in top
            else ("未命中", TIER_COST["未命中"]))

tier_stats = {}
N_QUERIES = 400
for _ in range(N_QUERIES):
    item = long_items[rng.integers(len(long_items))]
    age = int(rng.integers(1, 90))
    tier, cost = layered_retrieve(age, item)
    s = tier_stats.setdefault(tier, {"count": 0, "cost": 0.0})
    s["count"] += 1
    s["cost"] += cost

print(f"{'结果':<6s}{'查询占比':>10s}{'平均延迟':>12s}")
total_latency = sum(s["cost"] for s in tier_stats.values())
for tier in ["短期", "中期", "长期", "未命中"]:
    if tier in tier_stats:
        s = tier_stats[tier]
        print(f"{tier:<6s}{s['count'] / N_QUERIES:>9.1%}{s['cost'] / s['count']:>10.1f}ms")
print(f"\n整体平均检索延迟: {total_latency / N_QUERIES:.1f}ms")
print("观察：分层把'新近记忆'挡在廉价的内存层，只有冷数据才走向量库；")
print("长期层噪声会导致未命中——需要人工兜底，而不是无限加大 top_k。")

## 实验 4：可视化——ReAct 的补偿效应 + 记忆分层效果

**左图**：任务需要 n=1..5 步外部操作（查数据/计算/校验…）。
- 一步直答：每步可靠度 0.75 且无反馈，成功率随步数指数衰减；
- ReAct：行动产生的 **Observation** 提高单步可靠度（0.90），失败后还有一次凭观察修正的机会——这正是"边想边做"相对"一条路想到黑"的结构性优势。

**右图**：实验 3 的记忆分层命中率与单位延迟。

In [ ]:
n_steps_range = np.arange(1, 6)
N_TASKS = 3000
STEP_DIRECT, STEP_REACT, FIX_PROB = 0.75, 0.90, 0.60

r = np.random.default_rng(3)
acc_direct, acc_react = [], []
for n in n_steps_range:
    direct = np.ones(N_TASKS, dtype=bool)
    react = np.ones(N_TASKS, dtype=bool)
    for _ in range(n):
        direct &= r.random(N_TASKS) < STEP_DIRECT
        ok = r.random(N_TASKS) < STEP_REACT
        fixed = (~ok) & (r.random(N_TASKS) < FIX_PROB)   # Observation 带来的修正机会
        react &= ok | fixed
    acc_direct.append(direct.mean())
    acc_react.append(react.mean())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.8))

ax1.plot(n_steps_range, acc_direct, "o--", color="#999999", label="一步直答（无工具反馈）")
ax1.plot(n_steps_range, acc_react, "o-", color="#4c9f9a", label="ReAct（行动+观察修正）")
ax1.set_xlabel("任务需要的工具/推理步数")
ax1.set_ylabel("任务成功率")
ax1.set_ylim(0, 1.02)
ax1.set_title("ReAct：反馈循环对多步任务的补偿")
ax1.legend()
ax1.grid(alpha=0.3)

tiers = ["短期", "中期", "长期", "未命中"]
shares = [tier_stats.get(t, {"count": 0})["count"] / N_QUERIES for t in tiers]
costs = [TIER_COST[t] for t in tiers]
bars = ax2.bar(tiers, shares, color=["#4c9f9a", "#7fb3d5", "#f0b429", "#d96d73"], alpha=0.85)
for b, s, c in zip(bars, shares, costs):
    ax2.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.015,
             f"{s:.0%}\n{c:g}ms", ha="center", fontsize=10)
ax2.set_ylabel("查询占比")
ax2.set_ylim(0, max(max(shares) * 1.35, 0.1))
ax2.set_title(f"记忆分层检索：命中率与单位延迟（平均 {total_latency / N_QUERIES:.1f}ms）")
ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("本周六大误区对照：")
for m in ["Agent = ChatGPT + 插件？  → 它是决策-执行-记忆的完整系统",
          "工具描述随便写？          → Description 是 LLM 选工具的唯一依据",
          "Agent 越多越好？          → 3-5 个专业 Agent 通常优于 10+ 个",
          "CoT 万能？                → 只对推理类任务有明显提升",
          "参数错误也重试？          → 同输入同错误，应反馈修正",
          "日志以后再加？            → 可观测性必须在架构阶段考虑"]:
    print("  -", m)

## 一周一句话

> **Agent = LLM + Tools + Memory + ReAct。让 AI 从"只会说"变成"能做事"。**

自我检查（对照 md 清单）：
- [ ] Function Calling 三要素：Name + Description + Parameters（Description 最重要）
- [ ] ReAct 循环：Thought → Action → Observation，每步可调整方向
- [ ] 参数校验失败 ≠ 暂时性故障：不重试，反馈给 LLM 修正
- [ ] 记忆分层：对话用内存、用户用缓存、知识用数据库
- [ ] 生产三支柱：状态管理 + 错误处理 + 可观测性